In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [3]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [19]:
param_dict = {
    'short_window' : 40 ,
    'long_window' : 120 , 
    'threshold' : 0.1
}

In [ ]:
%%time
from collections import deque
import logging

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        self.position_last = 0

        self.short_window = param_dict.get('short_window')
        self.long_window = param_dict.get('long_window')
        self.threshold = param_dict.get('threshold',0.0)


        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
        )
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"策略初始化 - 短周期: {param_dict.get('short_window')}, 长周期: {param_dict.get('long_window')}, 阈值: {param_dict.get('threshold')}")

        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0

        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return

        self.price_list.append(price)

        if(len(self.price_list) < self.long_window):
            self.position_last = 0
            self.prev_signal = 0
            return

        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        long_ma = sum(self.price_list) / self.long_window

        if short_ma - long_ma > self.threshold:
            current_signal = 1
        elif short_ma - long_ma < -self.threshold:
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal:
            self.position_last = current_signal
            self.prev_signal = current_signal

            if current_signal == 1:
                self.logger.info(f"【买入信号】价格: {price}, 短均线: {short_ma:.4f}, 长均线: {long_ma:.4f}, 差值: {diff:.4f}")
            elif current_signal == -1:
                self.logger.info(f"【卖出信号】价格: {price}, 短均线: {short_ma:.4f}, 长均线: {long_ma:.4f}, 差值: {diff:.4f}")
            else:
                self.logger.info(f"【平仓/中性】价格: {price}, 均线回归中性区域")

        return

    

CPU times: user 13 μs, sys: 2 μs, total: 15 μs
Wall time: 16.7 μs


In [27]:
%%time
def backtest_demo(instrument_id, trade_ymd, strategy_name):
    strategy = StrategyDemo(param_dict)
    snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
    position_dict = {}
    for snap in snap_list[:]:
        strategy.on_snap(snap)
        position_dict[snap['time_mark']] = strategy.position_last
    profit = base_tool.backtest_quick(instrument_id, trade_ymd, strategy_name, position_dict)
    print(position_dict)

    return profit
profit = backtest_demo('511520', '20260319', 'simple_MA')


2026-03-27 06:07:28,853 - INFO - 策略初始化 - 短周期: 40, 长周期: 120, 阈值: 0.1


/home/jovyan/work/backtest_result/511520_20260319_simple_MA.pkl Please wait
无交易或结果错误，重新尝试。
{1773883800000: 0, 1773883801000: 0, 1773883802000: 0, 1773883803000: 0, 1773883804000: 0, 1773883805000: 0, 1773883806000: 0, 1773883807000: 0, 1773883808000: 0, 1773883809000: 0, 1773883810000: 0, 1773883811000: 0, 1773883812000: 0, 1773883813000: 0, 1773883814000: 0, 1773883815000: 0, 1773883816000: 0, 1773883817000: 0, 1773883818000: 0, 1773883819000: 0, 1773883820000: 0, 1773883821000: 0, 1773883822000: 0, 1773883823000: 0, 1773883824000: 0, 1773883825000: 0, 1773883826000: 0, 1773883827000: 0, 1773883828000: 0, 1773883829000: 0, 1773883830000: 0, 1773883831000: 0, 1773883832000: 0, 1773883833000: 0, 1773883834000: 0, 1773883835000: 0, 1773883836000: 0, 1773883837000: 0, 1773883838000: 0, 1773883839000: 0, 1773883840000: 0, 1773883841000: 0, 1773883842000: 0, 1773883843000: 0, 1773883844000: 0, 1773883845000: 0, 1773883846000: 0, 1773883847000: 0, 1773883848000: 0, 1773883849000: 0, 17738838